In [25]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))

In [ ]:
import pandas as pd
from pathlib import Path

androids_path = Path("../data/raw/androids")

In [26]:
control = pd.read_csv(androids_path / "bdi-control.csv")
depression = pd.read_csv(androids_path / "bdi-depression.csv")
folds_raw = pd.read_csv(androids_path / "fold-lists.csv")
timedata_raw = pd.read_csv(androids_path / "interview_timedata.csv", header=None)

In [27]:
print("CONTROL")
print(control.head())

print("\nDEPRESSION")
print(depression.head())

print("\nFOLDS")
print(folds_raw.head())

print("\nTIMEDATA")
print(timedata_raw.head())

CONTROL
            File   BDI
0  01_CF56_1.wav  24.0
1  02_CM57_2.wav   9.0
2  03_CF30_3.wav  15.0
3  04_CF57_3.wav  10.0
4  05_CF41_3.wav  16.0

DEPRESSION
            File BDI
0  01_PM58_2.wav  15
1  02_PM65_2.wav  23
2  03_PF66_3.wav   7
3  04_PF50_2.wav  52
4  05_PM53_4.wav  23

FOLDS
          Read   Unnamed: 1   Unnamed: 2   Unnamed: 3   Unnamed: 4  \
0        fold1        fold2        fold3        fold4        fold5   
1  '01_CF56_1'  '05_CF41_3'  '06_CF44_2'  '07_CF50_2'  '03_CF30_3'   
2  '02_CM57_2'  '11_CF44_2'  '15_CF53_3'  '10_CF51_2'  '04_CF57_3'   
3  '09_CF56_3'  '32_CF22_3'  '20_CM51_3'  '12_CF36_1'  '08_CF42_2'   
4  '21_CF58_3'  '46_CF39_3'  '23_CF55_3'  '14_CF49_3'  '13_CF45_2'   

   Unnamed: 5  Unnamed: 6    Interview   Unnamed: 8   Unnamed: 9  Unnamed: 10  \
0         NaN         NaN        fold1        fold2        fold3        fold4   
1         NaN         NaN  '01_CF56_1'  '12_CF36_1'  '03_CF30_3'  '05_CF41_3'   
2         NaN         NaN  '02_CM57_2'  '14_C

In [30]:
control = control.copy()
depression = depression.copy()

control.columns = [c.strip().lower() for c in control.columns]
depression.columns = [c.strip().lower() for c in depression.columns]

control["depressed"] = 0
depression["depressed"] = 1

labels = pd.concat([control, depression], ignore_index=True)
labels = labels.rename(columns={"file": "file", "bdi": "bdi_score"})

labels["file"] = labels["file"].astype(str).str.strip()
labels["file_stem"] = labels["file"].str.replace(".wav", "", regex=False)

labels.head()

,file,bdi_score,depressed,file_stem
0,01_CF56_1.wav,24.0,0,01_CF56_1
1,02_CM57_2.wav,9.0,0,02_CM57_2
2,03_CF30_3.wav,15.0,0,03_CF30_3
3,04_CF57_3.wav,10.0,0,04_CF57_3
4,05_CF41_3.wav,16.0,0,05_CF41_3


In [31]:
audio_dir = androids_path / "audio"

audio_files = list(audio_dir.glob("*.wav"))
audio_df = pd.DataFrame({"file_path": audio_files})
audio_df["file"] = audio_df["file_path"].apply(lambda x: x.name)
audio_df["file_stem"] = audio_df["file_path"].apply(lambda x: x.stem)

audio_df.head()

,file_path,file,file_stem


In [33]:
androids_meta = audio_df.merge(labels, on=["file", "file_stem"], how="left")
androids_meta.head()

,file_path,file,file_stem,bdi_score,depressed


In [34]:
folds_raw

,Read,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Interview,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,fold1,fold2,fold3,fold4,fold5,NaN,NaN,fold1,fold2,fold3,fold4,fold5
1,'01_CF56_1','05_CF41_3','06_CF44_2','07_CF50_2','03_CF30_3',NaN,NaN,'01_CF56_1','12_CF36_1','03_CF30_3','05_CF41_3','06_CF44_2'
2,'02_CM57_2','11_CF44_2','15_CF53_3','10_CF51_2','04_CF57_3',NaN,NaN,'02_CM57_2','14_CF49_3','04_CF57_3','11_CF44_2','07_CF50_2'
3,'09_CF56_3','32_CF22_3','20_CM51_3','12_CF36_1','08_CF42_2',NaN,NaN,'18_CM64_3','26_CM31_3','08_CF42_2','21_CF58_3','13_CF45_2'
4,'21_CF58_3','46_CF39_3','23_CF55_3','14_CF49_3','13_CF45_2',NaN,NaN,'20_CM51_3','33_CF46_3','09_CF56_3','32_CF22_3','16_CF33_4'
5,'22_CF50_3','47_CF61_2','26_CM31_3','16_CF33_4','17_CF55_3',NaN,NaN,'22_CF50_3','38_CM27_3','10_CF51_2','44_CF37_3','19_CF62_4'
6,'25_CF59_3','07_PM39_4','27_CF63_4','28_CF34_3','18_CM64_3',NaN,NaN,'23_CF55_3','46_CF39_3','15_CF53_3','01_PM58_2','27_CF63_4'
7,'33_CF46_3','13_PF58_2','30_CF62_3','29_CF34_3','19_CF62_4',NaN,NaN,'24_CM63_3','51_CF52_2','17_CF55_3','04_PF50_2','36_CF59_2'
8,'41_CM71_2','14_PF35_3','31_CF55_2','36_CF59_2','24_CM63_3',NaN,NaN,'25_CF59_3','52_CF29_3','28_CF34_3','07_PM39_4','39_CM27_3'
9,'55_CM29_3','15_PM63_4','37_CF69_1','42_CF53_4','38_CM27_3',NaN,NaN,'29_CF34_3','53_CF52_1','31_CF55_2','09_PM60_2','45_CF40_3'


In [35]:
folds_raw = pd.read_csv(androids_path / "fold-lists.csv", header=None)
folds_raw.head(10)

,0,1,2,3,4,5,6,7,8,9,10,11
0,Read,NaN,NaN,NaN,NaN,NaN,NaN,Interview,NaN,NaN,NaN,NaN
1,fold1,fold2,fold3,fold4,fold5,NaN,NaN,fold1,fold2,fold3,fold4,fold5
2,'01_CF56_1','05_CF41_3','06_CF44_2','07_CF50_2','03_CF30_3',NaN,NaN,'01_CF56_1','12_CF36_1','03_CF30_3','05_CF41_3','06_CF44_2'
3,'02_CM57_2','11_CF44_2','15_CF53_3','10_CF51_2','04_CF57_3',NaN,NaN,'02_CM57_2','14_CF49_3','04_CF57_3','11_CF44_2','07_CF50_2'
4,'09_CF56_3','32_CF22_3','20_CM51_3','12_CF36_1','08_CF42_2',NaN,NaN,'18_CM64_3','26_CM31_3','08_CF42_2','21_CF58_3','13_CF45_2'
5,'21_CF58_3','46_CF39_3','23_CF55_3','14_CF49_3','13_CF45_2',NaN,NaN,'20_CM51_3','33_CF46_3','09_CF56_3','32_CF22_3','16_CF33_4'
6,'22_CF50_3','47_CF61_2','26_CM31_3','16_CF33_4','17_CF55_3',NaN,NaN,'22_CF50_3','38_CM27_3','10_CF51_2','44_CF37_3','19_CF62_4'
7,'25_CF59_3','07_PM39_4','27_CF63_4','28_CF34_3','18_CM64_3',NaN,NaN,'23_CF55_3','46_CF39_3','15_CF53_3','01_PM58_2','27_CF63_4'
8,'33_CF46_3','13_PF58_2','30_CF62_3','29_CF34_3','19_CF62_4',NaN,NaN,'24_CM63_3','51_CF52_2','17_CF55_3','04_PF50_2','36_CF59_2'
9,'41_CM71_2','14_PF35_3','31_CF55_2','36_CF59_2','24_CM63_3',NaN,NaN,'25_CF59_3','52_CF29_3','28_CF34_3','07_PM39_4','39_CM27_3'


In [36]:
def extract_fold_section(df, col_start, col_end, speech_type):
    section = df.iloc[2:, col_start:col_end].copy()
    section.columns = df.iloc[1, col_start:col_end]
    section = section.reset_index(drop=True)

    long_df = section.melt(var_name="fold", value_name="file_stem")
    long_df["speech_type"] = speech_type
    long_df["file_stem"] = long_df["file_stem"].astype(str).str.strip()
    long_df = long_df[long_df["file_stem"].notna()]
    long_df = long_df[long_df["file_stem"] != ""]
    long_df = long_df[~long_df["file_stem"].str.lower().eq("nan")]
    return long_df

In [37]:
read_folds = extract_fold_section(folds_raw, 0, 5, "read")
interview_folds = extract_fold_section(folds_raw, 7, 12, "interview")

folds_tidy = pd.concat([read_folds, interview_folds], ignore_index=True)
folds_tidy.head()

,fold,file_stem,speech_type
0,fold1,'01_CF56_1',read
1,fold1,'02_CM57_2',read
2,fold1,'09_CF56_3',read
3,fold1,'21_CF58_3',read
4,fold1,'22_CF50_3',read


In [38]:
timedata_raw = pd.read_csv(androids_path / "interview_timedata.csv", header=None)

timedata_raw = timedata_raw.dropna(axis=1, how="all")
timedata_raw = timedata_raw.rename(columns={0: "file_stem"})

timedata_raw["file_stem"] = timedata_raw["file_stem"].astype(str).str.strip()
timedata_raw.head()

,file_stem,1,2,3,4,5,6,7,8,9,...,57,58,59,60,61,62,63,64,65,66
0,01_CF56_1,8.161,38.001,41.402,46.895,51.120,81.428,89.810,128.706,132.721,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,02_CM57_2,4.076,6.006,9.902,29.893,34.789,40.268,44.360,56.818,66.509,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,03_CF30_3,6.425,40.180,46.286,48.947,55.725,57.515,61.887,77.468,83.889,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,04_CF57_3,5.698,28.316,32.603,58.604,65.163,86.992,88.627,106.423,117.172,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,05_CF41_3,13.107,19.129,27.642,34.519,42.098,44.468,49.155,49.915,55.395,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
androids_meta = androids_meta.merge(folds_tidy, on="file_stem", how="left")

In [40]:
androids_meta = androids_meta.merge(timedata_raw, on="file_stem", how="left")

In [41]:
androids_meta.head()
androids_meta.columns.tolist()
androids_meta.shape

(0, 73)

In [42]:
androids_meta.to_csv("../data/processed/androids_metadata.csv", index=False)
folds_tidy.to_csv("../data/processed/androids_folds_tidy.csv", index=False)
timedata_raw.to_csv("../data/processed/androids_interview_timedata_clean.csv", index=False)

In [43]:
androids_core = androids_meta[
    ["file_path", "file", "file_stem", "bdi_score", "depressed", "fold", "speech_type"]
].copy()

androids_core.head()



,file_path,file,file_stem,bdi_score,depressed,fold,speech_type


In [44]:
androids_core.to_csv("../data/processed/androids_metadata_core.csv", index=False)